In [0]:
%sql
-- Registro de los indicadores de UNDP en la tabla de configuración (decisión D25).
-- Mismo criterio que Banco Mundial/WGI/ILOSTAT: una fila por indicador evaluado, con incluido y motivo.
-- MERGE con WHEN NOT MATCHED: si se vuelve a correr, no duplica ni pisa filas que ya existen.
-- 'codigo' es el prefijo de las columnas del archivo (hdi_1990, hdi_1991, ...). Silver lo usa para despivotar.
MERGE INTO happiness.meta.config_indicadores AS destino
USING (
  SELECT * FROM VALUES
    ('UNDP', 'hdi',   'Indice de Desarrollo Humano (IDH)',        'desarrollo humano', 'indice 0-1',        true,  'indicador central de UNDP; cobertura ~98% en 2011 y 2023', DATE '2026-11-26'),
    ('UNDP', 'le',    'Esperanza de vida al nacer (UNDP)',        'desarrollo humano', 'anios',             true,  'componente del IDH; cobertura 100%; solapa con SP.DYN.LE00.IN de Banco Mundial, evaluar redundancia (Spearman >= 0,95, D3) en Silver', DATE '2026-11-26'),
    ('UNDP', 'eys',   'Anios esperados de escolaridad',           'desarrollo humano', 'anios',             true,  'componente del IDH (educacion); cobertura ~99%', DATE '2026-11-26'),
    ('UNDP', 'mys',   'Anios medios de escolaridad',              'desarrollo humano', 'anios',             true,  'componente del IDH (educacion); cobertura ~98%', DATE '2026-11-26'),
    ('UNDP', 'gnipc', 'Ingreso nacional bruto per capita (UNDP)', 'desarrollo humano', 'USD PPA constantes', true,  'componente del IDH; cobertura ~99%; solapa con NY.GDP.PCAP.PP.KD de Banco Mundial, evaluar redundancia en Silver', DATE '2026-11-26'),
    ('UNDP', 'gii',   'Indice de Desigualdad de Genero',          'desarrollo humano', 'indice 0-1',        true,  'sexto indicador elegido: dimension de igualdad de genero que ningun otro cubre; cobertura ~79% en 2011 (cerca del umbral de D3), ~88% en 2023', DATE '2026-11-26'),
    ('UNDP', 'ihdi',  'IDH ajustado por desigualdad',             'desarrollo humano', 'indice 0-1',        false, 'cobertura ~67% en 2011, bajo el 80% de D3; ademas casi redundante con hdi', DATE '2026-11-26'),
    ('UNDP', 'gdi',   'Indice de Desarrollo de Genero',           'desarrollo humano', 'razon mujeres/hombres', false, 'candidato a sexto indicador, descartado a favor de gii; queda como reemplazo si gii no pasa D3 en Silver (cobertura ~89% en 2011)', DATE '2026-11-26')
  AS t(fuente, codigo, nombre, tema, unidad, incluido, motivo, fecha_decision)
) AS fuente_datos
ON destino.fuente = fuente_datos.fuente AND destino.codigo = fuente_datos.codigo
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Verificación: cuántos indicadores incluidos y descartados hay ahora por fuente.
-- Esperado: UNDP = 6 incluidos y 2 descartados; Banco Mundial 26 y 1; WGI 4 y 2; ILOSTAT 2 y 4.
SELECT fuente, incluido, COUNT(*) AS cantidad
FROM happiness.meta.config_indicadores
GROUP BY fuente, incluido
ORDER BY fuente, incluido DESC;